In [ ]:
#Beta testing and Development: "ML"
#Credit D. Pierre Lauray


import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib
from sklearn import tree
from sklearn import preprocessing
from sklearn import utils
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
# import dependencies and doc

mgm = pd.read_csv('mgm_grand_NBA.csv', encoding='latin1')


,Unnamed: 0,game_id,game_date,away_team,home_team,pregame_odds,total_over_points,total_over_stake_percentage,total_over_wager_percentage,total_over_odds,...,spread_away_decimal_odds,spread_away_stake_percentage,spread_away_wager_percentage,spread_away_won,spread_home_points,spread_home_odds,spread_home_decimal_odds,spread_home_stake_percentage,spread_home_wager_percentage,spread_home_won
0,0,nba.g.2021101913,2021-10-19-10:00,Golden State,LA Lakers,"-3.5, O/U 226.5",226.5,51.07,54.19,-110,...,1.91,78.16,81.95,True,-3.5,-110.0,1.91,21.84,18.05,False
1,0,nba.g.2021101915,2021-10-19-10:00,Brooklyn,Milwaukee,"-2, O/U 234",234.0,32.02,40.55,-110,...,1.87,42.13,47.62,False,-2.0,-105.0,1.95,57.87,52.38,True
2,0,nba.g.2021102003,2021-10-20-10:00,Philadelphia,New Orleans,"-3.5, O/U 224.5",224.5,61.52,65.97,-110,...,1.91,62.12,61.77,True,3.5,-110.0,1.91,37.88,38.23,False
3,0,nba.g.2021102008,2021-10-20-10:00,Chicago,Detroit,"-5, O/U 218",218.0,63.47,67.11,-110,...,1.91,68.77,67.44,True,5.0,-110.0,1.91,31.23,32.56,False
4,0,nba.g.2021102016,2021-10-20-10:00,Houston,Minnesota,"-6.5, O/U 232.5",232.5,69.82,56.40,-110,...,1.91,53.52,60.31,False,-6.5,-110.0,1.91,46.48,39.69,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6076,0,nba.g.2026021128,2026-02-11-10:00,Detroit,Toronto,"-1.5, O/U 223.5",223.5,91.82,91.00,-110,...,1.91,62.85,60.00,True,-1.5,-110.0,1.91,37.15,40.00,False
6077,0,nba.g.2026021130,2026-02-11-10:00,Atlanta,Charlotte,"-2.5, O/U 235.5",235.5,88.78,88.12,-110,...,1.87,47.31,44.30,False,-2.5,-105.0,1.95,52.69,55.70,True
6078,0,nba.g.2026021213,2026-02-12-10:00,Dallas,LA Lakers,"-6.5, O/U 237.5",237.5,78.42,79.85,-110,...,1.87,28.49,33.70,False,-6.5,-105.0,1.95,71.51,66.30,True
6079,0,nba.g.2026021225,2026-02-12-10:00,Milwaukee,Oklahoma City,"-13.5, O/U 214.5",214.5,97.42,93.83,-110,...,1.91,46.69,47.40,True,-13.5,-110.0,1.91,53.31,52.60,False


In [10]:
#cleaned preprocessing
# Fix target
y = mgm["total_over_won"]   # or total_under_won

# Clean pregame odds using vectorized operations
odds = (
    mgm["pregame_odds"]
    .str.replace(", O/U", ",", regex=True)
    .str.extract(r'(?P<over>[\d\.]+),\s*(?P<under>[\d\.]+)')
    .astype(float)
)
print(odds)
#X = mgm[["total_over_odds","total_under_odds","home_team","away_team"]].replace(team_values)
#X["pre_game_over"] = odds["over"]
#X["pre_game_under"] = odds["under"]

#X = X.fillna(X.mean())

# Train/test split
#X_train, X_test, y_train, y_test = train_test_split(
   # X, y, test_size=0.10, random_state=0, stratify=y
#)

# Model
#model = DecisionTreeClassifier()
#model.fit(X_train, y_train)

#preds = model.predict(X_test)
#score = accuracy_score(y_test, preds)

#print("Accuracy:", score)


      over  under
0      3.5  226.5
1      2.0  234.0
2      3.5  224.5
3      5.0  218.0
4      6.5  232.5
...    ...    ...
6076   1.5  223.5
6077   2.5  235.5
6078   6.5  237.5
6079  13.5  214.5
6080   6.5  236.5

[6081 rows x 2 columns]


In [ ]:
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# 1. Load & merge all your game-level data into one DataFrame `df_games`
#    Each row = one game, with:
#    - home_team, away_team, date
#    - market odds: spread, total, moneyline
#    - engineered features: rolling stats, defensive efficiency, etc.
#    - targets: y_over, y_under, y_ml_home, y_spread_home

feature_cols = [...]   # numeric + categorical feature names
target_cols  = ["y_over", "y_under", "y_ml_home", "y_spread_home"]

X = df_games[feature_cols]
y = df_games[target_cols]

# 2. Preprocess: numeric vs categorical
numeric_features = [...]   # e.g. odds, rolling efficiencies, etc.
categorical_features = [...]  # e.g. home_team, away_team, league, sport

preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

# 3. Base LightGBM classifier
base_lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary",
    n_jobs=-1
)

# 4. One model per target (simpler than MultiOutput at first)
models = {}
for target in target_cols:
    pipe = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", base_lgbm)
    ])
    models[target] = pipe


In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

for target, pipe in models.items():
    scores = cross_val_score(
        pipe,
        X,
        y[target],
        cv=tscv,
        scoring="roc_auc"  # or 'neg_log_loss' for probabilities
    )
    print(target, "CV AUC:", scores.mean(), "+/-", scores.std())

    # Fit on all past data (you can choose a cutoff date)
    pipe.fit(X, y[target])


In [ ]:
def suggest_bets(df_upcoming, threshold=0.55):
    X_new = df_upcoming[feature_cols]
    suggestions = []

    for target, pipe in models.items():
        proba = pipe.predict_proba(X_new)[:, 1]  # P(class=1)
        df_upcoming[target + "_proba"] = proba

    # Example: suggest bets where model probability beats implied odds
    for idx, row in df_upcoming.iterrows():
        game_suggestions = []

        # Example: over bet
        implied_over = row["implied_over_prob"]  # from market odds
        if row["y_over_proba"] > max(implied_over + 0.03, threshold):
            game_suggestions.append(("over", row["y_over_proba"], implied_over))

        # similarly for under, moneyline, spread...

        if game_suggestions:
            suggestions.append((idx, row["game_id"], game_suggestions))

    return suggestions


In [ ]:
import matplotlib.pyplot as plt
import lightgbm as lgb

for target, pipe in models.items():
    lgbm_model = pipe.named_steps["model"]
    # If using ColumnTransformer + OneHotEncoder, get feature names:
    # (you can extract them from preprocess.transformers_)

    lgb.plot_importance(lgbm_model, max_num_features=20)
    plt.title(f"Feature importance for {target}")
    plt.show()


You’re thinking about this the right way—what you want is not just “a model”, but a pipeline that’s:

sport‑agnostic

leak‑free (respecting time)

able to output probabilities for multiple bet types

easy to extend with new datasets

Let’s design that, then I’ll give you a concrete LightGBM‑based code skeleton you can start from.


1. Define the prediction targets
You want four main outputs per game:

Total over probability

Total under probability

Home moneyline win probability

Home spread cover probability

The cleanest way is to build one row per game with:

Features: pregame odds, team strength, recent form, etc.

Targets (binary):

y_over = 1 if final total > closing total line

y_under = 1 if final total < closing total line

y_ml_home = 1 if home team wins

y_spread_home = 1 if home team covers

Then you can either:

Train one model per target, or

Use MultiOutputClassifier(LGBMClassifier(...)) to wrap LightGBM for all four at once. 

I’d start with separate models—easier to debug and tune.

2. Build sport‑agnostic features
Think in terms of generic concepts, not NBA‑specific:

Team strength: rolling offensive/defensive efficiency, Elo, net rating

Form: last N games averages, trends

Context: home/away, rest days, back‑to‑back, playoffs vs regular season

Market info: closing spread, total, moneyline prices

From your defensive efficiency file, you already have team‑level defensive metrics like:

Okla City,1,1.044,1.21,1.036,1.052

You can merge these by team name and game date to create features like:

def_eff_season

def_eff_last3

def_eff_home / def_eff_away

From the playoff stats file, you have rich player totals (e.g. Shai Gilgeous‑Alexander’s line):

Shai Gilgeous-Alexander, PG, 26, OKC, 23, 23, 851, 233, 504, 0.462, ...

You can aggregate these to team‑level per‑game stats (e.g. team usage, scoring load, 3P rate) and then roll them over time.

Key idea: for any sport, you can always compute:

rolling team performance

rolling opponent‑adjusted metrics

market vs performance deltas

3. Avoiding data leakage (critical for betting)
For betting, time is everything. No shuffling, no random splits.

Sort games by date

Use time‑series splits (e.g. TimeSeriesSplit or manual folds)

For each game, compute features using only past games

Example pattern:

Train on seasons 2010–2018 → validate on 2019

Train on 2010–2019 → validate on 2020

etc.

Never use full‑season stats that include the game you’re predicting. All rolling stats must be computed causally (up to t-1).

4. LightGBM + scikit‑learn pipeline
LightGBM integrates cleanly with scikit‑learn, so you can use pipelines and cross‑validation directly. 

High‑level structure:

python
5. Time‑series cross‑validation and training
python
This gives you leak‑free CV and a trained model per bet type.

6. Getting bet suggestions with probabilities
Once models are trained, you can generate probabilities for upcoming games:

python
You’ll need to compute implied probabilities from odds (American, decimal, etc.) and compare them to your model’s probabilities to find +EV spots.

7. Visualizing key drivers (feature importance, SHAP)
With LightGBM, you can easily inspect feature importance:

python
For deeper insight, you can use SHAP to see how features push probabilities up or down for individual games.

8. Extending to other sports
To make this truly sport‑agnostic:

Keep a common schema at the game level:
sport, league, season, date, home_team, away_team, home_score, away_score, market_odds..., engineered_features..., targets...

For each sport, write a feature builder that outputs the same kind of columns:

rolling offensive/defensive strength

pace/tempo

home/away effects

market vs performance deltas

Then the same LightGBM pipelines can be reused; you just train on that sport’s data.

9. About the Kaggle historical data
That curl command will give you a large historical NBA dataset. The workflow:

Unzip and load all seasons into a single DataFrame.

Build game‑level rows (one per game).

Compute rolling team stats and merge with your MGM odds + defensive efficiency + playoff stats.

Train exactly as above with time‑series splits.

If you want, next step we can:

Design the exact schema for your unified game table

Write a feature‑engineering function for NBA specifically (rolling stats, merging your defensive efficiency and player totals)

Then plug it straight into the LightGBM pipeline above.

In [2]:
import kagglehub


# Download latest version
path = kagglehub.dataset_download("eoinamoore/historical-nba-data-and-player-box-scores")

print("Path to dataset files:", path)

100%|██████████| 884M/884M [01:33<00:00, 9.89MB/s] 

Extracting files...


Path to dataset files: C:\Users\15617\.cache\kagglehub\datasets\eoinamoore\historical-nba-data-and-player-box-scores\versions\443
